In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer
import torch
import pandas as pd
from datasets import Dataset

In [20]:
# Set the path to the file you'd like to load
fake_path = "Data/Fake.csv"
real_path = "Data/True.csv"
tweets_df_train_path = "Data/train.csv"
tweets_df_test_path = "Data/test.csv"

fake_df = pd.read_csv(fake_path)
fake_df['label'] = "fake"
real_df = pd.read_csv(real_path)
real_df['label'] = "real"

real_df['all_text'] = real_df['title'] + " " + real_df['text']
fake_df['all_text'] = fake_df['title'] + " " + fake_df['text']

df = pd.concat([fake_df, real_df], ignore_index=True)

label_mapping={0: 'real', 1: 'fake', 'Real': 0, 'Fake': 1}

df['label'] = df['label'].map(label_mapping)
df['all_text'] = df['all_text'].fillna("").astype(str)

In [21]:
model_id = "answerdotai/ModernBERT-base"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

def tokenize_function(examples):
    return tokenizer(
        examples["all_text"], 
        padding="max_length", 
        truncation=True, 
        max_length=1024
    )

hf_dataset = Dataset.from_pandas(df)

# 2. Apply the tokenization
# (batched=True is crucial here so it processes chunks of text at once)
tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)

split_datasets = tokenized_datasets.train_test_split(test_size=0.2, seed=42)

Loading weights: 100%|██████████| 136/136 [00:00<00:00, 4876.81it/s]
ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 52511/52511 [00:19<00:00, 2727.05 examples/s]


In [25]:
print(df['label'].unique())

<ArrowStringArray>
['fake', 'real']
Length: 2, dtype: str


In [23]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./modernbert-fake-news",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    bf16=True # Keep True if using an Ampere/newer GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_datasets["train"],
    eval_dataset=split_datasets["test"],
    compute_metrics=compute_metrics,
)

In [24]:
trainer.train()

/Users/jackfogerty/Documents/Coding/FakeNewsDetection/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


ValueError: too many dimensions 'str'

In [ ]:
article_text = "Breaking: Scientists have just discovered that the moon is entirely made of cheese."

# Prepare the text
inputs = tokenizer(article_text, return_tensors="pt").to(model.device)

# Get predictions
with torch.no_grad():
    outputs = model(**inputs)
    
prediction = outputs.logits.argmax(dim=-1).item()

# Map the output back to a human-readable label
if prediction == 1:
    print("Classification: Fake News")
else:
    print("Classification: Real News")